In [6]:
#Install required dependencies
#!pip install fastapi uvicorn transformers torch nest-asyncio requests

In [7]:
# SQLite database configuration
import sqlite3
from datetime import datetime

DB_NAME = "chatbot_logs.db"

def init_db():
    """Initializes the database and creates the logs table if it doesn't exist."""
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS logs (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            timestamp TEXT,
            user_message TEXT,
            bot_response TEXT
        )
    ''')
    conn.commit()
    conn.close()

def log_interaction(user_message, bot_response):
    """Inserts a user query and the chatbot's response into the database."""
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    cursor.execute(
        "INSERT INTO logs (timestamp, user_message, bot_response) VALUES (?, ?, ?)",
        (timestamp, user_message, bot_response)
    )
    conn.commit()
    conn.close()

# Initialize the database file
init_db()
print("Database initialized successfully.")

Database initialized successfully.


In [8]:
import sys
!{sys.executable} -m pip install fastapi uvicorn transformers torch pydantic nest-asyncio requests

Defaulting to user installation because normal site-packages is not writeable


In [16]:
# ==========================================
# 1. IMPORTS
# ==========================================
import sqlite3
from datetime import datetime
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# ==========================================
# 2. SQLITE DATABASE LOGGING CONFIGURATION
# ==========================================
DB_NAME = "chatbot_logs.db"

def init_db():
    """Creates the SQLite database table for transaction logging."""
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS logs (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            timestamp TEXT,
            user_message TEXT,
            bot_response TEXT
        )
    ''')
    conn.commit()
    conn.close()

def log_interaction(user_message, bot_response):
    """Logs a single conversation turn into the database file."""
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    cursor.execute(
        "INSERT INTO logs (timestamp, user_message, bot_response) VALUES (?, ?, ?)",
        (timestamp, user_message, bot_response)
    )
    conn.commit()
    conn.close()

# Initialize database
init_db()

# ==========================================
# 3. NLP ENGINE INITIALIZATION
# ==========================================
MODEL_NAME = "microsoft/DialoGPT-medium"
print("Loading NLP Model weights... (This might take a moment on the first run)")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

# Track conversational history context tokens globally
chat_history_ids = None

# ==========================================
# 4. DIRECT INTERACTION FUNCTION
# ==========================================
def get_chatbot_response(user_message: str):
    """Processes messages directly through the model and logs them to SQLite."""
    global chat_history_ids
    user_input = user_message.strip()
    
    if not user_input:
        return {"error": "Message field cannot be empty."}
        
    try:
        # Encode user string input and concatenate with history
        new_user_input_ids = tokenizer.encode(user_input + tokenizer.eos_token, return_tensors='pt')
        
        if chat_history_ids is not None:
            bot_input_ids = torch.cat([chat_history_ids, new_user_input_ids], dim=-1)
        else:
            bot_input_ids = new_user_input_ids

        # Generate bot response tokens based on context history
        chat_history_ids = model.generate(
            bot_input_ids, 
            max_length=1000, 
            pad_token_id=tokenizer.eos_token_id
        )
        
        # Decode output tokens back into string text
        bot_response = tokenizer.decode(chat_history_ids[:, bot_input_ids.shape[-1]:][0], skip_special_tokens=True)
        
        # Log to SQLite
        log_interaction(user_input, bot_response)
        
        return {"user": user_input, "bot": bot_response}
        
    except Exception as e:
        return {"error": f"NLP Processing Exception: {str(e)}"}

print("\n🎉 Setup Complete! Model and logging database are fully loaded.")

Loading NLP Model weights... (This might take a moment on the first run)


Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]


🎉 Setup Complete! Model and logging database are fully loaded.


In [17]:
import requests

url = "http://127.0.0.1:8000/chat"
payload = {"message": "Hello! Can you help me check my support ticket?"}

try:
    response = requests.post(url, json=payload)
    print("Server Status:", response.status_code)
    print("Chatbot Reply:", response.json())
except Exception as e:
    print("Error details:", e)

Server Status: 200
Chatbot Reply: {'user': 'Hello! Can you help me check my support ticket?', 'bot': "Sure , I'll help you out ."}
